Single table column transformation


In [0]:
dbutils.fs.ls("abfss://bronze@intechsacc.dfs.core.windows.net/SalesLT /")

In [0]:
dbutils.fs.ls("abfss://silver@intechsacc.dfs.core.windows.net/")

In [0]:
df = spark.read.format("parquet").load(
    "abfss://bronze@intechsacc.dfs.core.windows.net/SalesLT / Address/Address.parquet"
)

In [0]:
display(df)

In [0]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

df = df.withColumn("ModifiedDate", date_format(from_utc_timestamp(df['ModifiedDate'].cast(TimestampType()), "UTC"), "yyyy-MM-dd"))

In [0]:
display(df)

In [0]:
%sql
SELECT 1 as colum1

Data transformation for tables

In [0]:
table_name = []

for i in dbutils.fs.ls("abfss://bronze@intechsacc.dfs.core.windows.net/SalesLT /"):
    table_name.append(i.name.split('/')[0])

table_name

In [0]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

for i in table_name:
    path = (
        "abfss://bronze@intechsacc.dfs.core.windows.net/"
        "SalesLT /" + i + "/"
    )
    
    df = spark.read.format('parquet').load(path)
    column = df.columns

    for col in column:
        if "Date" in col or "date" in col:
            df = df.withColumn(col, date_format(from_utc_timestamp(df[col].cast(TimestampType()), "UTC"), "yyyy-MM-dd"))
    output_path = "abfss://silver@intechsacc.dfs.core.windows.net/SalesLT /" + i + "/"
    df.write.format("delta").mode('overwrite').save(output_path)

In [0]:
display(df)